# Host the AI Skin Analysis API on Google Colab
This notebook will start a Flask server (the same `server.py` you have locally) and expose it via `ngrok` so that your web app can call it from anywhere.

## Step 1: Install Dependencies

In [ ]:
!pip install flask pyngrok ultralytics -q

## Step 2: Upload `server.py` and the trained model zip
1. In the Files panel (left side) click the folder icon.
2. Drag & drop `server.py` from `d:/facial-skin-analysis/src/` and `trained_model.zip` (the zip that contains `best.pt`).

## Step 3: Unzip the model and start the Flask app

In [ ]:
import os, zipfile, subprocess, sys, time, signal, json, base64, pathlib

# Unzip the model
if os.path.exists('trained_model.zip'):
    with zipfile.ZipFile('trained_model.zip', 'r') as zip_ref:
        zip_ref.extractall('model_files')
    print('Model unzipped into model_files/')
else:
    print('⚠️ trained_model.zip not found – make sure you uploaded it.')

# Ensure server.py is present
if not os.path.isfile('server.py'):
    raise FileNotFoundError('server.py not found in the root. Upload it.')

# Start Flask in a background process
# We use gunicorn via the command line for better ngrok compatibility
os.system('nohup gunicorn -b 0.0.0.0:5000 server:app &')
print('✅ Flask server launching on port 5000…')

# Wait a couple seconds for the server to be ready
time.sleep(4)

# Expose via ngrok
from pyngrok import ngrok
public_url = ngrok.connect(5000, bind_tls=True).public_url
print(f'🔗 Public URL: {public_url}')

# Save the public URL to a file so you can copy it later
with open('ngrok_url.txt', 'w') as f:
    f.write(public_url)
print('URL saved to ngrok_url.txt')

## Step 4: Grab the URL and update your web app
Open the file `ngrok_url.txt` (you'll see it in the Files panel). Copy the URL and paste it into your `app.js` where the placeholder `API_ENDPOINT` is located.